In [1]:
"""Load training data collected from Kafka prices topic."""
import pandas as pd

df = pd.read_csv("training_data.csv")
print(f"Loaded {len(df)} windows")
print(f"Symbols: {df['symbol'].unique().tolist()}")
df.head()

Loaded 50 windows
Symbols: ['BTCUSDT', 'ETHUSDT']


,symbol,window,trades,avg_price,volatility,min_price,max_price,price_range
0,BTCUSDT,2026-06-02T21:40:00,5495,66756.8671,25.5618,66709.20,66800.00,90.80
1,BTCUSDT,2026-06-02T21:41:00,5239,66670.7995,24.8202,66638.93,66737.59,98.66
2,BTCUSDT,2026-06-02T21:42:00,2561,66713.3210,34.3635,66656.19,66771.71,115.52
3,BTCUSDT,2026-06-02T21:43:00,7928,66862.8046,41.2589,66771.71,66918.00,146.29
4,BTCUSDT,2026-06-02T21:44:00,4658,66934.4608,17.2290,66902.00,66964.01,62.01


In [2]:
"""Inspect data distribution by symbol."""
df.groupby("symbol").agg({
    "trades":     ["count", "mean"],
    "volatility": ["mean", "min", "max"],
    "avg_price":  "mean",
}).round(2)

trades          volatility              avg_price
         count     mean       mean   min    max      mean
symbol                                                   
BTCUSDT     25  5617.40      26.53  8.34  50.82  67106.88
ETHUSDT     25  5226.28       0.80  0.18   1.64   1907.10

In [3]:
"""
Feature set for the model. avg_price is excluded because BTC (~67000)
and ETH (~1900) live on different scales — a shared model would be
dominated by BTC. We train one IsolationForest per symbol instead.
"""
features = ["volatility", "price_range", "trades"]
print(df[features].describe().round(2))

       volatility  price_range   trades
count       50.00        50.00    50.00
mean        13.67        48.86  5421.84
std         15.80        55.00  1922.12
min          0.18         0.79  1767.00
25%          0.71         2.60  4239.50
50%          4.99        18.49  5305.00
75%         25.38        91.70  6441.50
max         50.82       159.86  9572.00


In [4]:
"""
Train one IsolationForest per symbol with contamination=0.1.
With only 25 windows per symbol we expect ~2-3 to be flagged as anomalies.
"""
from sklearn.ensemble import IsolationForest

models = {}

for symbol in df["symbol"].unique():
    sub = df[df["symbol"] == symbol]
    if len(sub) < 10:
        print(f"Skipping {symbol}: only {len(sub)} samples")
        continue

    X = sub[features].values

    model = IsolationForest(
        contamination=0.1,
        random_state=42,
        n_estimators=100,
    )
    model.fit(X)

    scores = model.decision_function(X)
    preds = model.predict(X)
    n_anom = (preds == -1).sum()

    print(f"{symbol}: trained on {len(sub)} windows, "
          f"flagged {n_anom} anomalies "
          f"(score range: {scores.min():.3f} to {scores.max():.3f})")

    models[symbol] = model

BTCUSDT: trained on 25 windows, flagged 3 anomalies (score range: -0.067 to 0.116)
ETHUSDT: trained on 25 windows, flagged 3 anomalies (score range: -0.109 to 0.150)


In [5]:
"""Inspect which windows the model considers anomalous."""
for symbol, model in models.items():
    sub = df[df["symbol"] == symbol].copy()
    X = sub[features].values
    sub["score"] = model.decision_function(X)
    sub["is_anomaly"] = model.predict(X) == -1

    flagged = sub[sub["is_anomaly"]].sort_values("score")
    print(f"\n=== {symbol} — flagged windows ===")
    print(flagged[["window", "trades", "volatility",
                   "price_range", "score"]].to_string(index=False))


=== BTCUSDT — flagged windows ===
             window  trades  volatility  price_range     score
2026-06-02T21:57:00    1778      8.3370        31.22 -0.066908
2026-06-02T21:50:00    9572     50.8158       159.86 -0.053832
2026-06-02T21:42:00    2561     34.3635       115.52 -0.006153

=== ETHUSDT — flagged windows ===
             window  trades  volatility  price_range     score
2026-06-02T21:57:00    1767      0.1827         0.79 -0.109423
2026-06-02T21:47:00    6486      1.6423         5.76 -0.081522
2026-06-02T21:50:00    8428      1.4427         4.63 -0.007713


In [6]:
"""Save dict of models for the Flask API to load in Day 4."""
import joblib

joblib.dump(models, "model.pkl")
print(f"Saved {len(models)} models to model.pkl")

# Verify
loaded = joblib.load("model.pkl")
print(f"Loaded back: {list(loaded.keys())}")

Saved 2 models to model.pkl
Loaded back: ['BTCUSDT', 'ETHUSDT']


In [7]:
"""Simulate what the Flask API will do."""
import numpy as np

def score_window(symbol, volatility, price_range, trades):
    if symbol not in models:
        return "UNKNOWN", 0.0
    X = np.array([[volatility, price_range, trades]])
    pred = models[symbol].predict(X)[0]
    score = float(models[symbol].decision_function(X)[0])
    label = "ANOMALY" if pred == -1 else "NORMAL"
    return label, score

print("Normal BTC window:")
print(score_window("BTCUSDT", volatility=15.0, price_range=50.0, trades=2500))

print("\nHigh volatility BTC window:")
print(score_window("BTCUSDT", volatility=200.0, price_range=1500.0, trades=500))

print("\nNormal ETH window:")
print(score_window("ETHUSDT", volatility=0.4, price_range=1.5, trades=2000))

print("\nHigh volatility ETH window:")
print(score_window("ETHUSDT", volatility=10.0, price_range=50.0, trades=500))

Normal BTC window:
('NORMAL', 0.08340848316845634)

High volatility BTC window:
('ANOMALY', -0.10500025422913695)

Normal ETH window:
('NORMAL', 0.047714439413371945)

High volatility ETH window:
('ANOMALY', -0.10934333389411666)
